# Winner Ensemble (Standard + MSD)

Este notebook treina dois pipelines em um unico submit, fazendo um ensemble leve por media ponderada das probabilidades de teste.

Componentes:

1. BERTimbau standard (reproducao exata do winner), com head de classificacao padrao do `AutoModelForSequenceClassification`.
2. BERTimbau com Multi-Sample Dropout (MSD), mesma base, mas com cinco dropouts amostrados e media dos logits.

Cada arquitetura e treinada em cross validation estratificado com cinco folds, totalizando dez modelos.

As probabilidades OOF sao usadas para busca simples do peso otimo do ensemble. A calibracao fixa do winner (temperatura e thresholds por classe) e aplicada no blend final antes de gerar a submissao.

Meta: ganho de 0.3 a 0.8 pontos percentuais em F1 macro contra o winner isolado, vindo da diversidade de arquiteturas. Tempo estimado na GPU T4 do Kaggle: cerca de seis horas. O enunciado da competicao exige analise comparativa da indicacao clinica nos laudos mamograficos, e o preprocessamento preserva essa marcacao com acentos.

In [ ]:
import os
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

MODEL_PATH = '/kaggle/input/models/fabianofilho/bertimbau-ptbr-complete/pytorch/default/1'
TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

MAX_LEN     = 512
BATCH_SIZE  = 8
EPOCHS      = 5
LR          = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
N_FOLDS     = 5
SEED        = 42
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
NUM_CLASSES = 7

BEST_TEMP       = 0.958
BEST_THRESHOLDS = [0.95, 1.6, 1.35, 1.0, 0.4, 1.15, 0.1]

MSD_N_SAMPLES = 5
MSD_DROPOUT   = 0.5

WEIGHT_STANDARD = 0.5
WEIGHT_MSD      = 0.5

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicacao:']
ACHADOS_MARKERS     = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'analise comparativa:']

def extract_section(text, start_markers, end_markers=None):
    txt_lower = text.lower()
    start_idx = -1
    for m in start_markers:
        idx = txt_lower.find(m)
        if idx >= 0:
            start_idx = idx + len(m)
            break
    if start_idx < 0:
        return ''
    if end_markers is None:
        return text[start_idx:].strip()
    end_idx = len(text)
    for m in end_markers:
        idx = txt_lower.find(m, start_idx)
        if 0 < idx < end_idx:
            end_idx = idx
    return text[start_idx:end_idx].strip()

def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def build_input_text(report_text):
    report      = clean_text(report_text)
    indicacao   = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados     = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if indicacao:   parts.append(f'Indicação: {indicacao}')
    if achados:     parts.append(f'Achados: {achados}')
    if comparativa: parts.append(f'Comparativa: {comparativa}')
    return ' '.join(parts)

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print('Train:', train_df.shape, 'Test:', test_df.shape)

train_texts  = [build_input_text(t) for t in train_df['report'].fillna('').tolist()]
train_labels = train_df['target'].tolist()
test_texts   = [build_input_text(t) for t in test_df['report'].fillna('').tolist()]
print('Exemplo train_text[0]:')
print(train_texts[0][:300])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

class SPRDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=MAX_LEN,
            return_tensors='pt',
        )
        item = {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'token_type_ids': enc.get('token_type_ids', torch.zeros_like(enc['input_ids'])).squeeze(0),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class FocalLoss(nn.Module):
    def __init__(self, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
    def forward(self, logits, target):
        logp = F.log_softmax(logits, dim=-1)
        p = logp.exp()
        logp_t = logp.gather(1, target.unsqueeze(1)).squeeze(1)
        p_t    = p.gather(1, target.unsqueeze(1)).squeeze(1)
        loss = -self.alpha * (1 - p_t) ** self.gamma * logp_t
        return loss.mean()

class MSDBertClassifier(nn.Module):
    def __init__(self, model_path, num_classes=NUM_CLASSES, n_samples=MSD_N_SAMPLES, dropout=MSD_DROPOUT):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_path, local_files_only=True)
        hidden = self.bert.config.hidden_size
        self.dropouts = nn.ModuleList([nn.Dropout(dropout) for _ in range(n_samples)])
        self.classifier = nn.Linear(hidden, num_classes)
        self.n_samples = n_samples
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        pooled = out.last_hidden_state[:, 0, :]
        logits = None
        for dp in self.dropouts:
            l = self.classifier(dp(pooled))
            logits = l if logits is None else logits + l
        return logits / self.n_samples

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, scaler, criterion, is_msd=False):
    model.train()
    total = 0.0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attn      = batch['attention_mask'].to(device)
        tti       = batch['token_type_ids'].to(device)
        labels    = batch['labels'].to(device)
        optimizer.zero_grad()
        with autocast():
            if is_msd:
                logits = model(input_ids=input_ids, attention_mask=attn, token_type_ids=tti)
            else:
                out = model(input_ids=input_ids, attention_mask=attn, token_type_ids=tti)
                logits = out.logits
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += loss.item()
    return total / max(1, len(loader))

@torch.no_grad()
def predict_standard(model, loader):
    model.eval()
    probs = []
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attn      = batch['attention_mask'].to(device)
        tti       = batch['token_type_ids'].to(device)
        with autocast():
            out = model(input_ids=input_ids, attention_mask=attn, token_type_ids=tti)
        probs.append(F.softmax(out.logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(probs, axis=0)

@torch.no_grad()
def predict_msd(model, loader):
    model.eval()
    for dp in model.dropouts:
        dp.train()
    probs = []
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attn      = batch['attention_mask'].to(device)
        tti       = batch['token_type_ids'].to(device)
        with autocast():
            logits = model(input_ids=input_ids, attention_mask=attn, token_type_ids=tti)
        probs.append(F.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(probs, axis=0)

def compute_f1(probs, labels):
    preds = np.argmax(probs, axis=1)
    return f1_score(labels, preds, average='macro')

def temperature_scale(probs, temperature):
    eps = 1e-12
    logits = np.log(np.clip(probs, eps, 1.0))
    logits = logits / temperature
    logits = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(logits)
    return e / e.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thresholds):
    w = np.array(thresholds, dtype=np.float64)
    adj = probs * w[None, :]
    return np.argmax(adj, axis=1)

In [ ]:
set_seed(SEED)
oof_std  = np.zeros((len(train_texts), NUM_CLASSES))
test_std = np.zeros((len(test_texts),  NUM_CLASSES))
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

test_ds = SPRDataset(test_texts)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_texts, train_labels)):
    print(f'[standard] fold {fold+1}/{N_FOLDS}')
    tr_texts  = [train_texts[i] for i in tr_idx]
    tr_labels = train_labels[tr_idx]
    va_texts  = [train_texts[i] for i in va_idx]
    va_labels = train_labels[va_idx]

    tr_ds = SPRDataset(tr_texts, tr_labels)
    va_ds = SPRDataset(va_texts, va_labels)
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, num_labels=NUM_CLASSES, local_files_only=True
    ).to(device)

    no_decay = ['bias', 'LayerNorm.weight']
    params = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': WEIGHT_DECAY},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
    ]
    optimizer = torch.optim.AdamW(params, lr=LR)
    total_steps = len(tr_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
    scaler = GradScaler()
    criterion = FocalLoss()

    best_f1 = -1.0
    best_val_probs = None
    best_test_probs = None
    for epoch in range(EPOCHS):
        loss = train_epoch(model, tr_loader, optimizer, scheduler, scaler, criterion, is_msd=False)
        val_probs = predict_standard(model, va_loader)
        f1 = compute_f1(val_probs, va_labels)
        print(f'  epoch {epoch+1} loss={loss:.4f} val_f1={f1:.5f}')
        if f1 > best_f1:
            best_f1 = f1
            best_val_probs = val_probs
            best_test_probs = predict_standard(model, test_loader)

    oof_std[va_idx]  = best_val_probs
    test_std        += best_test_probs / N_FOLDS
    print(f'[standard] fold {fold+1} best_f1={best_f1:.5f}')

    del model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
set_seed(SEED)
oof_msd  = np.zeros((len(train_texts), NUM_CLASSES))
test_msd = np.zeros((len(test_texts),  NUM_CLASSES))
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_texts, train_labels)):
    print(f'[msd] fold {fold+1}/{N_FOLDS}')
    tr_texts  = [train_texts[i] for i in tr_idx]
    tr_labels = train_labels[tr_idx]
    va_texts  = [train_texts[i] for i in va_idx]
    va_labels = train_labels[va_idx]

    tr_ds = SPRDataset(tr_texts, tr_labels)
    va_ds = SPRDataset(va_texts, va_labels)
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = MSDBertClassifier(MODEL_PATH).to(device)

    no_decay = ['bias', 'LayerNorm.weight']
    params = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': WEIGHT_DECAY},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
    ]
    optimizer = torch.optim.AdamW(params, lr=LR)
    total_steps = len(tr_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
    scaler = GradScaler()
    criterion = FocalLoss()

    best_f1 = -1.0
    best_val_probs = None
    best_test_probs = None
    for epoch in range(EPOCHS):
        loss = train_epoch(model, tr_loader, optimizer, scheduler, scaler, criterion, is_msd=True)
        val_probs = predict_msd(model, va_loader)
        f1 = compute_f1(val_probs, va_labels)
        print(f'  epoch {epoch+1} loss={loss:.4f} val_f1={f1:.5f}')
        if f1 > best_f1:
            best_f1 = f1
            best_val_probs = val_probs
            best_test_probs = predict_msd(model, test_loader)

    oof_msd[va_idx]  = best_val_probs
    test_msd        += best_test_probs / N_FOLDS
    print(f'[msd] fold {fold+1} best_f1={best_f1:.5f}')

    del model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
oof_std_cal = temperature_scale(oof_std, BEST_TEMP)
oof_std_preds = apply_thresholds(oof_std_cal, BEST_THRESHOLDS)
f1_std = f1_score(train_labels, oof_std_preds, average='macro')

oof_msd_cal = temperature_scale(oof_msd, BEST_TEMP)
oof_msd_preds = apply_thresholds(oof_msd_cal, BEST_THRESHOLDS)
f1_msd = f1_score(train_labels, oof_msd_preds, average='macro')

print(f'OOF F1 standard: {f1_std:.5f}')
print(f'OOF F1 MSD:      {f1_msd:.5f}')

best_w = 0.5
best_f1 = 0.0
for w in np.arange(0.1, 1.0, 0.1):
    blend = w * oof_std + (1 - w) * oof_msd
    cal = temperature_scale(blend, BEST_TEMP)
    preds = apply_thresholds(cal, BEST_THRESHOLDS)
    f1 = f1_score(train_labels, preds, average='macro')
    print(f'  w_std={w:.2f} f1={f1:.5f}')
    if f1 > best_f1:
        best_f1, best_w = f1, w

print(f'Best weight standard: {best_w:.2f}, OOF F1 blend: {best_f1:.5f}')

In [ ]:
test_blend = best_w * test_std + (1 - best_w) * test_msd
test_cal = temperature_scale(test_blend, BEST_TEMP)
test_preds = apply_thresholds(test_cal, BEST_THRESHOLDS)
print('test_preds shape:', test_preds.shape)

In [ ]:
submission = pd.DataFrame({'ID': test_df['ID'], 'target': test_preds})
submission.to_csv('submission.csv', index=False)
print(submission['target'].value_counts().sort_index())
print('saved submission.csv with', len(submission), 'rows')